# 02 - Local upload to Google Earth Engine with `geeup`

This notebook prepares and uploads local shadow GeoTIFFs to a Google Earth Engine `ImageCollection` **using `geeup` directly from a local folder**.

The workflow is:

1. validate local TIFFs and parse their timestamps from filenames  
2. optionally sanitize filenames with `geeup rename`  
3. generate the `geeup` metadata CSV with `geeup getmeta`  
4. run a `geeup upload` dry-run  
5. run the real upload from the local folder to Earth Engine  
6. patch Earth Engine asset metadata (`start_time`, `height_tag`, `local_time`, etc.) from filenames  
7. validate the resulting collection

> **Important:** `geeup` still requires Earth Engine authentication and upload cookies / auth setup outside the upload command itself. Run the one-time authentication steps before the actual upload.


## 1. Install dependencies

Run this cell only if the current environment does not already contain the required packages.

This notebook uses:
- `earthengine-api` for metadata patching and validation
- `pandas` for manifest handling
- `geeup` for local batch upload to Earth Engine


In [ ]:
# %pip install earthengine-api pandas geeup


## 2. Import the libraries used in this notebook

The code below loads:
- standard Python utilities for paths, subprocess calls and time parsing
- `pandas` for tabular inspection
- the Earth Engine Python API for collection management and metadata patching


In [ ]:
from __future__ import annotations

import re
import shutil
import subprocess
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

import pandas as pd
import ee
from zoneinfo import ZoneInfo


## 3. Configure the upload workflow

Set:
- the Earth Engine project
- the destination `ImageCollection`
- the local input folder containing the TIFFs
- the timezone used to interpret filenames
- the upload account email used by `geeup`

The notebook supports these filename styles:

Preferred:
- `shadow_YYYYMMDD_HHMM_hXpYm.tif`

Legacy:
- `shadow_YYYYMMDD_HHMM.tif`
- `shadow_YYYYMMDD_HHMM_.tif`
- `shadow_YYYYMMDD_HHMM_ped.tif`


In [ ]:
@dataclass
class GeeupUploadConfig:
    gee_project: str = "YOUR_GEE_PROJECT"
    asset_collection: str = "projects/YOUR_GEE_PROJECT/assets/shadow_input_collection"

    # Local folder with the TIFFs to upload.
    input_dir: str = "path/to/local/tifs"

    # geeup / authentication
    geeup_user_email: str = "your_account@email.com"
    geeup_executable: str = "geeup"

    # Time interpretation
    timezone_name: str = "Europe/Rome"

    # File handling
    nodata_value: int = 255
    pyramiding_policy: str = "MODE"   # suitable for categorical 3-bit rasters
    workers: int = 2
    run_batch_rename: bool = False
    overwrite_existing_assets: bool = False
    resume_upload: bool = True
    retry_failed_only: bool = False
    dry_run_first: bool = True

    # Allowed: None, "standard", "pedestrian"
    # - standard accepts shadow_YYYYMMDD_HHMM.tif and shadow_YYYYMMDD_HHMM_.tif
    # - pedestrian accepts shadow_YYYYMMDD_HHMM_ped.tif
    legacy_mode: str | None = None

    # Metadata CSV generated by geeup getmeta
    metadata_csv: str = "geeup_metadata.csv"

    create_collection_if_missing: bool = True

cfg = GeeupUploadConfig()
cfg


## 4. Validate the local environment

This block checks that:
- `geeup` is installed and visible on PATH
- the Earth Engine CLI is available if needed by your local environment
- the local input folder exists

It also prints the one-time authentication commands that should already be completed before you launch a real upload.


In [ ]:
def validate_executable(name: str) -> str:
    exe = shutil.which(name)
    if exe is None:
        raise RuntimeError(f"Executable not found on PATH: {name}")
    return exe


def validate_local_environment(cfg: GeeupUploadConfig) -> None:
    geeup_path = validate_executable(cfg.geeup_executable)
    print(f"geeup executable: {geeup_path}")

    # Not always strictly required by every setup, but useful to check.
    ee_cli = shutil.which("earthengine")
    print(f"earthengine CLI: {ee_cli if ee_cli else 'not found on PATH'}")

    input_dir = Path(cfg.input_dir).resolve()
    if not input_dir.exists():
        raise FileNotFoundError(f"Input folder not found: {input_dir}")
    print(f"Input folder: {input_dir}")

    print("\nOne-time authentication steps to run outside the notebook if needed:")
    print("  earthengine authenticate")
    print("  geeup cookie_setup")
    print("Optional service account setup:")
    print("  geeup auth --cred /path/to/service-account.json")


validate_local_environment(cfg)


## 5. Initialize Earth Engine

Earth Engine is used here for:
- creating the destination collection if needed
- patching uploaded asset metadata
- validating the uploaded collection after the `geeup` step


In [ ]:
def initialize_earth_engine(gee_project: str, force_auth: bool = False) -> None:
    if not gee_project or "YOUR_" in gee_project:
        raise ValueError("Set a valid value in cfg.gee_project before continuing.")

    if force_auth:
        ee.Authenticate()

    ee.Initialize(project=gee_project)
    print(f"Earth Engine initialized on project: {gee_project}")


# Example:
# initialize_earth_engine(cfg.gee_project, force_auth=False)


## 6. Parse shadow filenames and derive upload metadata

This block extracts:
- local timestamp
- UTC `start_time`
- observer height tag
- observer height in meters

These values are **not** used by `geeup getmeta` directly; they are used later to patch the uploaded Earth Engine assets with the project-specific metadata required by the aggregation notebook.


In [ ]:
def parse_shadow_filename(
    filename: str,
    timezone_name: str = "Europe/Rome",
    legacy_mode: str | None = None,
) -> dict | None:
    preferred = re.match(
        r"shadow_(\d{4})(\d{2})(\d{2})_(\d{2})(\d{2})_(h[\dp]+m)\.tif$",
        filename,
    )
    if preferred:
        year, month, day = map(int, preferred.group(1, 2, 3))
        hour, minute = map(int, preferred.group(4, 5))
        height_tag = preferred.group(6)
    else:
        legacy_standard = re.match(
            r"shadow_(\d{4})(\d{2})(\d{2})_(\d{2})(\d{2})(?:_)?\.tif$",
            filename,
        )
        legacy_ped = re.match(
            r"shadow_(\d{4})(\d{2})(\d{2})_(\d{2})(\d{2})_ped\.tif$",
            filename,
        )

        if legacy_standard and legacy_mode == "standard":
            year, month, day = map(int, legacy_standard.group(1, 2, 3))
            hour, minute = map(int, legacy_standard.group(4, 5))
            height_tag = "h1p5m"
        elif legacy_ped and legacy_mode == "pedestrian":
            year, month, day = map(int, legacy_ped.group(1, 2, 3))
            hour, minute = map(int, legacy_ped.group(4, 5))
            height_tag = "h0p1m"
        else:
            return None

    local_tz = ZoneInfo(timezone_name)
    dt_local = datetime(year, month, day, hour, minute, tzinfo=local_tz)
    dt_utc = dt_local.astimezone(timezone.utc)

    observer_height_m = float(height_tag[1:-1].replace("p", "."))

    return {
        "filename": filename,
        "stem": Path(filename).stem,
        "height_tag": height_tag,
        "observer_height_m": observer_height_m,
        "local_time": dt_local.strftime("%Y-%m-%dT%H:%M:%S"),
        "timezone_name": timezone_name,
        "epoch_ms": int(dt_utc.timestamp() * 1000),
        "start_time_rfc3339": dt_utc.strftime("%Y-%m-%dT%H:%M:%SZ"),
    }


## 7. Build and inspect the local manifest

This block scans the local TIFF folder, validates filenames, and builds a manifest table.

This is the main sanity check before upload:
- if a file does not appear here, it will not be uploaded correctly
- the expected Earth Engine asset IDs are also shown


In [ ]:
def list_local_candidates(cfg: GeeupUploadConfig) -> list[Path]:
    input_dir = Path(cfg.input_dir).resolve()
    return sorted(
        [p for p in input_dir.iterdir() if p.is_file() and p.suffix.lower() in {".tif", ".tiff"}],
        key=lambda p: p.name.lower(),
    )


def build_local_manifest(cfg: GeeupUploadConfig) -> pd.DataFrame:
    files = list_local_candidates(cfg)
    rows = []

    for tif in files:
        props = parse_shadow_filename(
            tif.name,
            timezone_name=cfg.timezone_name,
            legacy_mode=cfg.legacy_mode,
        )
        if props is None:
            continue

        rows.append({
            "filename": tif.name,
            "local_path": str(tif.resolve()),
            "asset_id_expected": f"{cfg.asset_collection}/{tif.stem}",
            **props,
        })

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["local_time", "filename"]).reset_index(drop=True)
    return df


manifest_df = build_local_manifest(cfg)
print(f"Compatible files found: {len(manifest_df)}")
manifest_df.head(20)


## 8. Optionally sanitize filenames with `geeup rename`

Earth Engine uploads are safer when filenames contain:
- no spaces
- no special characters
- simple stems that can become valid asset IDs

Set `cfg.run_batch_rename = True` only if you want the notebook to rename files automatically.
After renaming, rebuild the manifest.


In [ ]:
def run_command(cmd: list[str], check: bool = True) -> subprocess.CompletedProcess:
    print("Running:")
    print("  " + " ".join(cmd))
    return subprocess.run(cmd, check=check, text=True, capture_output=True)


def maybe_run_geeup_rename(cfg: GeeupUploadConfig) -> None:
    if not cfg.run_batch_rename:
        print("Skipping geeup rename (cfg.run_batch_rename = False).")
        return

    cmd = [
        cfg.geeup_executable,
        "rename",
        "--input",
        str(Path(cfg.input_dir).resolve()),
        "--batch",
    ]
    proc = run_command(cmd)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)


# Example:
# maybe_run_geeup_rename(cfg)
# manifest_df = build_local_manifest(cfg)
# manifest_df.head()


## 9. Create the `geeup` metadata CSV with `geeup getmeta`

`geeup getmeta` generates the metadata CSV needed by `geeup upload`, including fields such as:
- `system:index`
- raster dimensions
- number of bands
- data type
- inferred raster kind.

This cell runs `geeup getmeta` against the local raster folder.


In [ ]:
def generate_geeup_metadata(cfg: GeeupUploadConfig) -> Path:
    metadata_path = Path(cfg.metadata_csv).resolve()
    cmd = [
        cfg.geeup_executable,
        "getmeta",
        "--input",
        str(Path(cfg.input_dir).resolve()),
        "--metadata",
        str(metadata_path),
    ]
    proc = run_command(cmd)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if not metadata_path.exists():
        raise FileNotFoundError(f"geeup metadata CSV was not created: {metadata_path}")
    return metadata_path


# Example:
# metadata_csv_path = generate_geeup_metadata(cfg)


## 10. Preview the generated `geeup` metadata

This block loads the CSV generated by `geeup getmeta` so you can quickly inspect:
- image IDs (`system:index`)
- raster size
- number of bands
- inferred kind

If the stems here do not match the stems in `manifest_df`, the later metadata patch step will not align correctly.


In [ ]:
def preview_geeup_metadata(metadata_csv_path: str | Path) -> pd.DataFrame:
    path = Path(metadata_csv_path).resolve()
    if not path.exists():
        raise FileNotFoundError(f"Metadata CSV not found: {path}")
    df = pd.read_csv(path)
    return df


# Example:
# metadata_preview_df = preview_geeup_metadata(cfg.metadata_csv)
# metadata_preview_df.head(20)


## 11. Ensure the destination `ImageCollection` exists

`geeup` can often create missing collections interactively, but this helper keeps the workflow explicit and repeatable by creating the destination collection from Python if needed.


In [ ]:
def ensure_image_collection(asset_collection: str) -> None:
    try:
        ee.data.getAsset(asset_collection)
        print(f"ImageCollection already exists: {asset_collection}")
        return
    except Exception:
        pass

    ee.data.createAsset({"type": "IMAGE_COLLECTION"}, asset_collection)
    print(f"Created ImageCollection: {asset_collection}")


# Example:
# initialize_earth_engine(cfg.gee_project, force_auth=False)
# if cfg.create_collection_if_missing:
#     ensure_image_collection(cfg.asset_collection)


## 12. Run a `geeup upload` dry-run

`geeup` support `--dry-run` for validation without actually uploading. Use this before the real upload to catch configuration issues early.


In [ ]:
def build_geeup_upload_command(cfg: GeeupUploadConfig, dry_run: bool = False) -> list[str]:
    cmd = [
        cfg.geeup_executable,
        "upload",
        "--source",
        str(Path(cfg.input_dir).resolve()),
        "--dest",
        cfg.asset_collection,
        "--metadata",
        str(Path(cfg.metadata_csv).resolve()),
        "--user",
        cfg.geeup_user_email,
        "--nodata",
        str(cfg.nodata_value),
        "--pyramids",
        cfg.pyramiding_policy,
        "--workers",
        str(cfg.workers),
    ]

    if cfg.overwrite_existing_assets:
        cmd.extend(["--overwrite", "yes"])
    if cfg.resume_upload:
        cmd.append("--resume")
    if cfg.retry_failed_only:
        cmd.append("--retry-failed")
    if dry_run:
        cmd.append("--dry-run")

    return cmd


def run_geeup_dry_run(cfg: GeeupUploadConfig) -> None:
    cmd = build_geeup_upload_command(cfg, dry_run=True)
    proc = run_command(cmd, check=False)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    print(f"geeup dry-run exit code: {proc.returncode}")


# Example:
# if cfg.dry_run_first:
#     run_geeup_dry_run(cfg)


## 13. Run the real local upload with `geeup`

This is the main upload step from the local raster folder to the target Earth Engine `ImageCollection`.

`geeup` uploads from a local `--source` directory using:
- `--dest`
- `--metadata`
- `--user`
- optional `--workers`, `--resume`, `--nodata`, etc.

Run this cell only after:
- checking the local manifest
- generating the metadata CSV
- confirming authentication is ready


In [ ]:
def run_geeup_upload(cfg: GeeupUploadConfig) -> None:
    cmd = build_geeup_upload_command(cfg, dry_run=False)
    proc = run_command(cmd, check=False)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)

    if proc.returncode != 0:
        raise RuntimeError(f"geeup upload failed with exit code {proc.returncode}")


# Example:
# run_geeup_upload(cfg)


## 14. Optional: inspect current `geeup` / Earth Engine task status

This helper uses `geeup tasks` to check whether uploads are:
- running
- ready
- completed
- failed

Use it after launching the upload if you want a quick status query from the notebook.


In [ ]:
def run_geeup_tasks(state: str | None = None) -> None:
    cmd = ["geeup", "tasks"]
    if state:
        cmd.extend(["--state", state])
    proc = run_command(cmd, check=False)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)


# Examples:
# run_geeup_tasks()
# run_geeup_tasks("RUNNING")
# run_geeup_tasks("FAILED")


## 15. Patch uploaded Earth Engine asset metadata from filenames

`geeup getmeta` prepares upload-oriented metadata, but the aggregation workflow also needs project-specific fields derived from the filename, especially:
- `start_time`
- `height_tag`
- `observer_height_m`
- `local_time`
- `timezone_name`
- `source_filename`

This cell scans the uploaded assets in the target collection and patches them using `ee.data.updateAsset(...)`.


In [ ]:
def list_assets_in_collection(asset_collection: str) -> list[dict]:
    response = ee.data.listAssets(asset_collection, {"view": "FULL"})
    return response.get("assets", [])


def patch_collection_metadata_from_filenames(cfg: GeeupUploadConfig) -> list[str]:
    manifest_df = build_local_manifest(cfg)
    if manifest_df.empty:
        raise RuntimeError("No valid local TIFFs were found in the input folder.")

    by_stem = {row["stem"]: row for _, row in manifest_df.iterrows()}

    assets = list_assets_in_collection(cfg.asset_collection)
    if not assets:
        raise RuntimeError(
            "No assets found in the target image collection. "
            "Run geeup upload first, then rerun this cell."
        )

    patched_ids = []

    for asset in assets:
        asset_id = asset["name"]
        stem = asset_id.split("/")[-1]
        row = by_stem.get(stem)
        if row is None:
            print(f"⚠️  Skip asset without matching local filename: {asset_id}")
            continue

        new_metadata = {
            "start_time": row["start_time_rfc3339"],
            "properties": {
                "height_tag": row["height_tag"],
                "observer_height_m": float(row["observer_height_m"]),
                "local_time": row["local_time"],
                "timezone_name": row["timezone_name"],
                "source_filename": row["filename"],
            },
        }

        ee.data.updateAsset(
            asset_id,
            new_metadata,
            ["start_time", "properties"],
        )
        patched_ids.append(asset_id)
        print(f"✅ Patched metadata: {asset_id}")

    return patched_ids


# Example:
# patched = patch_collection_metadata_from_filenames(cfg)
# print(f"Assets patched: {len(patched)}")


## 16. Validate the final Earth Engine collection

This final check confirms that:
- images are present in the collection
- `height_tag` metadata is available
- sample `local_time` values can be read back

This is a quick test before moving to the aggregation notebook.


In [ ]:
def validate_collection(cfg: GeeupUploadConfig, sample_size: int = 5) -> None:
    collection = ee.ImageCollection(cfg.asset_collection)
    n_total = collection.size().getInfo()
    print(f"Total images in collection: {n_total}")

    for height_tag in ("h1p5m", "h0p1m"):
        n = collection.filter(ee.Filter.eq("height_tag", height_tag)).size().getInfo()
        print(f"  {height_tag}: {n}")

    sample = collection.limit(sample_size)
    props = sample.aggregate_array("local_time").getInfo()
    print(f"Sample local_time values: {props}")


# Example:
# validate_collection(cfg)


## 17. Recommended execution order

Run the notebook in this order:

1. set `cfg`
2. validate local environment
3. initialize Earth Engine
4. build and inspect `manifest_df`
5. optionally run `geeup rename`
6. generate the `geeup` metadata CSV
7. run the dry-run
8. run the real upload
9. wait until uploads are complete
10. patch asset metadata
11. validate the collection


In [ ]:
# Example sequence:
#
# validate_local_environment(cfg)
# initialize_earth_engine(cfg.gee_project, force_auth=False)
#
# if cfg.create_collection_if_missing:
#     ensure_image_collection(cfg.asset_collection)
#
# manifest_df = build_local_manifest(cfg)
# display(manifest_df.head(20))
#
# maybe_run_geeup_rename(cfg)
# manifest_df = build_local_manifest(cfg)
#
# metadata_csv_path = generate_geeup_metadata(cfg)
# display(preview_geeup_metadata(metadata_csv_path).head(20))
#
# if cfg.dry_run_first:
#     run_geeup_dry_run(cfg)
#
# run_geeup_upload(cfg)
#
# # Optional status checks while upload tasks are progressing:
# # run_geeup_tasks("RUNNING")
# # run_geeup_tasks("FAILED")
#
# patched = patch_collection_metadata_from_filenames(cfg)
# print(f"Assets patched: {len(patched)}")
#
# validate_collection(cfg)
